# E-commerce Customer Segmentation and Prediction
## BIA Capstone — complete notebook
The 20% customer holdout is isolated until the final test section.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import joblib

## 1. Load and clean

In [ ]:
df=pd.read_csv('../data.csv',encoding='latin1')
print('Raw shape:',df.shape)
df=df.drop_duplicates().copy()
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])
df=df.dropna(subset=['CustomerID']).copy()
df=df[(df.Quantity>0)&(df.UnitPrice>0)].copy()
df['CustomerID']=df.CustomerID.astype(int)
df['Revenue']=df.Quantity*df.UnitPrice
print('Clean shape:',df.shape)

## 2. Customer-level 80/20 split — lock the test set

In [ ]:
customers=df.CustomerID.unique()
train_customers,test_customers=train_test_split(customers,test_size=.20,random_state=42)
train_df=df[df.CustomerID.isin(train_customers)].copy()
test_df=df[df.CustomerID.isin(test_customers)].copy()
print(train_df.CustomerID.nunique(),test_df.CustomerID.nunique())
# Do not use test_df below until the final test section.

## 3. RFM feature engineering on development data

In [ ]:
reference_date=train_df.InvoiceDate.max()+pd.Timedelta(days=1)
def make_rfm(tx):
 return tx.groupby('CustomerID').agg(Recency=('InvoiceDate',lambda x:(reference_date-x.max()).days),Frequency=('InvoiceNo','nunique'),Monetary=('Revenue','sum')).reset_index()
rfm=make_rfm(train_df); features=['Recency','Frequency','Monetary']
Xlog=np.log1p(rfm[features]); scaler=StandardScaler(); X=scaler.fit_transform(Xlog)
rfm.describe()

## 4. Compare clustering algorithms

In [ ]:
rows=[]; models={}
for k in range(2,11):
 km=KMeans(n_clusters=k,n_init=10,random_state=42); labels=km.fit_predict(X); models[k]=km
 rows.append([k,km.inertia_,silhouette_score(X,labels),davies_bouldin_score(X,labels)])
metrics=pd.DataFrame(rows,columns=['k','inertia','silhouette','davies_bouldin']);display(metrics)
# K=4 is selected for business interpretability: it separates recent/occasional, at-risk, regular/growing and VIP/loyal customers.

In [ ]:
selected_k=4
kmeans=models[selected_k]; rfm['Cluster']=kmeans.labels_
print(rfm.groupby('Cluster')[features].mean().round(2))
print('K-Means silhouette:',silhouette_score(X,rfm.Cluster))
h=AgglomerativeClustering(n_clusters=selected_k).fit_predict(X)
print('Hierarchical silhouette:',silhouette_score(X,h))
# DBSCAN example
db=DBSCAN(eps=.55,min_samples=5).fit_predict(X); mask=db!=-1
if len(set(db[mask]))>=2: print('DBSCAN silhouette:',silhouette_score(X[mask],db[mask]))

## 5. Classification model

In [ ]:
Xa,Xv,ya,yv=train_test_split(Xlog,rfm.Cluster,test_size=.20,random_state=42,stratify=rfm.Cluster)
models_cls={'Logistic Regression':LogisticRegression(max_iter=5000,C=30,random_state=42),'Decision Tree':DecisionTreeClassifier(max_depth=8,min_samples_leaf=3,random_state=42),'Random Forest':RandomForestClassifier(n_estimators=150,random_state=42),'Gradient Boosting':GradientBoostingClassifier(random_state=42)}
for name,m in models_cls.items():
 m.fit(Xa,ya); p=m.predict(Xv); print(name,accuracy_score(yv,p),f1_score(yv,p,average='weighted'))
final_classifier=LogisticRegression(max_iter=5000,C=30,random_state=42).fit(Xlog,rfm.Cluster)

## 6. Final 20% test — first use of test_df

In [ ]:
test_rfm=make_rfm(test_df); test_log=np.log1p(test_rfm[features]); test_scaled=scaler.transform(test_log)
test_clusters=kmeans.predict(test_scaled); test_predictions=final_classifier.predict(test_log)
print('Held-out K-Means silhouette:',silhouette_score(test_scaled,test_clusters))
print('Classifier agreement with frozen K-Means:',accuracy_score(test_clusters,test_predictions))

## 7. Save models

In [ ]:
joblib.dump(scaler,'../models/rfm_scaler.joblib')
joblib.dump(kmeans,'../models/kmeans_model.joblib')
joblib.dump(final_classifier,'../models/customer_segment_classifier.joblib')
print('Saved.')